In [ ]:
!pip install backtesting

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from backtesting import Backtest, Strategy
import matplotlib.pyplot as plt

In [ ]:
# pobieranie danych
ticker = "MSFT"
df = yf.download(ticker, start="2020-01-01", end="2024-05-07", interval="1d")

# naprawianie multiindexu od yfinance
df.columns = df.columns.get_level_values(0)

In [ ]:
# obliczanie SMA i RSI
def calculate_rsi(data, window):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['RSI'] = calculate_rsi(df['Close'], window=14)

In [ ]:
# ustawianie zmienności, sprawdzanie czy cena wzrosła
df['Volatility'] = df['Close'].rolling(window=14).std()
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df.dropna(inplace=True)

In [ ]:
features = ['SMA_10', 'SMA_50', 'RSI', 'Volatility', 'Close']
X = df[features]
y = df['Target']

In [ ]:
# ustawianie zbioru treningowego i testowego względem daty z polecenia
split_date = "2024-01-01"
mask_train = X.index < split_date
mask_test = (df.index >= "2024-01-01") & (df.index <= "2024-05-06")
X_train, y_train = X.loc[mask_train], y.loc[mask_train]

In [ ]:
# siatka parametrów
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

In [ ]:
# inicjacja i trening modelu
xgb = XGBClassifier(eval_metric='logloss', use_label_encoder=False)
tscv = TimeSeriesSplit(n_splits=3)
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=tscv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print(f"Najlepsze parametry: {grid_search.best_params_}")

In [ ]:
# Backtesting
class ML_Strategy(Strategy):
    def init(self):
        self.model = best_model
    def next(self):
        # pobieranie obecnego dnia i predykcja
        current_features = self.data.df.iloc[-1][features].values.reshape(1, -1)
        prediction = self.model.predict(current_features)[0]
        # logika wejścia i wyjścia
        if prediction == 1:
            if not self.position:
                self.buy() # buy jeśli nie ma pozycji
        else:
            if self.position:
                self.position.close() # zamknięcie pozycji

In [ ]:
# Przygotowanie danych i backtest
test_data = df.loc[mask_test].copy()
bt = Backtest(test_data, ML_Strategy, cash=10000)
stats = bt.run()

print("Wyniki:")
print(stats)
# wykresik
bt.plot(open_browser=False)